# 04B Study memory through MCP

**Problem:** A student wants a learning assistant to remember a project goal and notes between sessions. Build real MCP tools that store, update and retrieve fictional study records from SQLite. Reconnect using a new server/client and prove that the records remain.

**Core build: about 40 minutes using the supplied starter.** The database file belongs to this notebook runtime; download it yourself if you want to keep it after Colab disconnects. This is structured task memory, not a vector database, knowledge graph, multi-user service or model-generated memory.

**First success:** Goal G-MCP-01 remains complete and its note remains readable after a different MCP server object connects to the same SQLite file.

**Reference:** Inspired by [AI Engineering Hub / Agent with MCP Memory](https://github.com/patchy631/ai-engineering-hub/tree/042aeb5067384981eeba6a07fa882f436bb1660d/agent-with-mcp-memory). This independently written classroom adaptation keeps the idea of memory behind MCP, using a small SQLite store rather than Graphiti, a search provider or an LLM. MCP means **Model Context Protocol**.


## 1 Prepare the MCP packages

Use a fresh CPU runtime. This cell installs only the pinned MCP packages if they are not already available; it never requests a model reply.

In [ ]:
# Install only the two MCP packages used by this lab, if needed.
import importlib.metadata as metadata
import subprocess, sys
required = {"fastmcp": "2.14.5", "mcp": "1.30.0"}
missing = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Setup ready:", {name: metadata.version(name) for name in required})
print("CPU lab: no model account, API key, GPU or public server is needed.")


## 2 Create a small persistent store

Read the two table definitions. Existing fictional goals and notes are retained when this cell runs again. Keeping a file across server reconnects differs from keeping it after a Colab runtime is deleted.

In [ ]:
# A small project-local SQLite file is this lab's persistent memory.
import json, re, sqlite3
from pathlib import Path
from fastmcp import FastMCP, Client
WORK_DIR = Path.cwd() / "bgs_day2_mcp"
WORK_DIR.mkdir(exist_ok=True)
MEMORY_PATH = WORK_DIR / "study_memory.sqlite3"
with sqlite3.connect(MEMORY_PATH) as database:
    database.execute("CREATE TABLE IF NOT EXISTS goals (goal_id TEXT PRIMARY KEY, title TEXT NOT NULL, topic TEXT NOT NULL, status TEXT NOT NULL)")
    database.execute("CREATE TABLE IF NOT EXISTS notes (goal_id TEXT NOT NULL, note TEXT NOT NULL, UNIQUE(goal_id, note))")
print("Persistent study memory ready:", MEMORY_PATH.name)
print("Existing records are kept. Use fictional learning goals, not private personal information.")


## 3 Expose memory as MCP capabilities

Inspect where the tools validate IDs and where they write SQLite rows. No tool accepts an arbitrary filename or SQL statement. The store is for a single classroom runtime.

In [ ]:
def build_study_server(database_path):
    server = FastMCP("BGS Study Memory")

    def rows(sql, parameters=()):
        with sqlite3.connect(database_path) as database:
            database.row_factory = sqlite3.Row
            return [dict(row) for row in database.execute(sql, parameters).fetchall()]

    @server.tool
    def remember_goal(goal_id: str, title: str, topic: str) -> dict:
        """Create or update a fictional study goal. An existing completion status is retained."""
        title, topic = title.strip(), topic.strip().upper()
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,24}", goal_id) or not 1 <= len(title) <= 120 or not 1 <= len(topic) <= 30:
            return {"ok": False, "error": "USE_VALID_ID_TITLE_AND_TOPIC"}
        existed = bool(rows("SELECT goal_id FROM goals WHERE goal_id = ?", (goal_id,)))
        with sqlite3.connect(database_path) as database:
            database.execute("INSERT INTO goals VALUES (?, ?, ?, 'planned') ON CONFLICT(goal_id) DO UPDATE SET title = excluded.title, topic = excluded.topic", (goal_id, title, topic))
        return {"ok": True, "created": not existed, "goal": rows("SELECT * FROM goals WHERE goal_id = ?", (goal_id,))[0]}

    @server.tool
    def list_goals(topic: str = "") -> dict:
        """Read saved goals; optionally filter the exact topic."""
        topic = topic.strip().upper()
        result = rows("SELECT * FROM goals WHERE (? = '' OR topic = ?) ORDER BY goal_id", (topic, topic))
        return {"ok": True, "goals": result, "count": len(result)}

    @server.tool
    def complete_goal(goal_id: str) -> dict:
        """Mark an existing goal complete. Unknown goal IDs do not change records."""
        if not rows("SELECT goal_id FROM goals WHERE goal_id = ?", (goal_id,)):
            return {"ok": False, "error": "GOAL_NOT_FOUND"}
        with sqlite3.connect(database_path) as database:
            database.execute("UPDATE goals SET status = 'complete' WHERE goal_id = ?", (goal_id,))
        return {"ok": True, "goal": rows("SELECT * FROM goals WHERE goal_id = ?", (goal_id,))[0]}

    @server.tool
    def add_learning_note(goal_id: str, note: str) -> dict:
        """Save one note for a known goal. Repeating the same note does not duplicate it."""
        note = note.strip()
        if not rows("SELECT goal_id FROM goals WHERE goal_id = ?", (goal_id,)):
            return {"ok": False, "error": "GOAL_NOT_FOUND"}
        if not 1 <= len(note) <= 280:
            return {"ok": False, "error": "USE_NOTE_1_TO_280_CHARACTERS"}
        with sqlite3.connect(database_path) as database:
            database.execute("INSERT OR IGNORE INTO notes VALUES (?, ?)", (goal_id, note))
        return {"ok": True, "goal_id": goal_id, "notes": rows("SELECT note FROM notes WHERE goal_id = ? ORDER BY note", (goal_id,))}

    @server.tool
    def get_learning_notes(goal_id: str) -> dict:
        """Read notes for a known goal."""
        if not rows("SELECT goal_id FROM goals WHERE goal_id = ?", (goal_id,)):
            return {"ok": False, "error": "GOAL_NOT_FOUND", "notes": []}
        return {"ok": True, "goal_id": goal_id, "notes": rows("SELECT note FROM notes WHERE goal_id = ? ORDER BY note", (goal_id,))}

    @server.resource("memory://study-guide", mime_type="application/json")
    def study_guide() -> str:
        return json.dumps({"purpose": "fictional learning goals", "memory_store": "project-local SQLite",
            "scope": "one classroom runtime; not a multi-user login system", "practice": "Use short goals and evidence-based notes."}, indent=2)

    @server.prompt
    def daily_update(topic: str) -> str:
        """Prepare a daily-update template from supplied goal observations."""
        return f"Using only the saved {topic} goals and notes supplied by the client, write done, next and one question. Add no invented progress."

    return server

study_server = build_study_server(MEMORY_PATH)
study_client = Client(study_server)
print("Study-memory MCP server and client ready.")


## 4 Discover tools and read context

Use MCP client discovery, read the study-guide resource and retrieve the daily-update prompt. No model is called and the template does not invent a summary.

In [ ]:
async with study_client:
    memory_tools = await study_client.list_tools()
    memory_resources = await study_client.list_resources()
    memory_prompts = await study_client.list_prompts()
    guide = await study_client.read_resource("memory://study-guide")
    daily_template = await study_client.get_prompt("daily_update", {"topic": "MCP"})
print("Tools:", [tool.name for tool in memory_tools])
print(guide[0].text)
print("Prompt template:", daily_template.messages[0].content.text)
assert {"remember_goal", "list_goals", "complete_goal", "add_learning_note", "get_learning_notes"}.issubset({tool.name for tool in memory_tools})
assert any(prompt.name == "daily_update" for prompt in memory_prompts)


## 5 Save a goal and a learning note

Running this cell intentionally saves the fictional demo records. Updates use the same goal ID, retain completion state, and avoid duplicate identical notes on rerun.

In [ ]:
# Running this cell intentionally saves a fictional demo goal and note.
DEMO_GOAL_ID = "G-MCP-01"
DEMO_NOTE = "Tools run operations; resources return context."
async with study_client:
    remembered = (await study_client.call_tool("remember_goal", {
        "goal_id": DEMO_GOAL_ID, "title": "Build a real MCP practice server", "topic": "MCP"})).data
    noted = (await study_client.call_tool("add_learning_note", {"goal_id": DEMO_GOAL_ID, "note": DEMO_NOTE})).data
    completed = (await study_client.call_tool("complete_goal", {"goal_id": DEMO_GOAL_ID})).data
print("Saved through MCP:", remembered)
print("Notes:", noted)
print("Current goal:", completed)
assert remembered["ok"] and noted["ok"] and completed["goal"]["status"] == "complete"
assert sum(row["note"] == DEMO_NOTE for row in noted["notes"]) == 1


## 6 Prove persistence after reconnecting

A new server object and a new client open the same file. Read the saved goal and note through MCP. This checks persistence rather than reusing the original Python object.

In [ ]:
# Create a different server object that opens the same SQLite file.
restarted_server = build_study_server(MEMORY_PATH)
restarted_client = Client(restarted_server)
assert restarted_server is not study_server
async with restarted_client:
    after_restart = (await restarted_client.call_tool("list_goals", {"topic": "MCP"})).data
    notes_after_restart = (await restarted_client.call_tool("get_learning_notes", {"goal_id": DEMO_GOAL_ID})).data
saved_goal = next(goal for goal in after_restart["goals"] if goal["goal_id"] == DEMO_GOAL_ID)
print("Read through a new server/client:", saved_goal)
print("Notes still present:", notes_after_restart)
assert saved_goal["status"] == "complete"
assert any(row["note"] == DEMO_NOTE for row in notes_after_restart["notes"])


## 7 Inspect validation and idempotence

An unknown goal ID must leave the other rows unchanged. Saving an identical note twice must not duplicate it. These are ordinary checks, not planted defects.

In [ ]:
# Check that an unknown ID cannot change another stored goal.
async with restarted_client:
    before = (await restarted_client.call_tool("list_goals", {})).data
    rejected = (await restarted_client.call_tool("complete_goal", {"goal_id": "UNKNOWN-GOAL"})).data
    after = (await restarted_client.call_tool("list_goals", {})).data
    repeated_note = (await restarted_client.call_tool("add_learning_note", {
        "goal_id": DEMO_GOAL_ID, "note": DEMO_NOTE})).data
print("Expected rejected operation:", rejected)
assert rejected["ok"] is False and rejected["error"] == "GOAL_NOT_FOUND"
assert before == after
assert sum(row["note"] == DEMO_NOTE for row in repeated_note["notes"]) == 1
print("Unknown-ID and repeated-note checks passed.")


## 8 Your own learning record

Task 1: replace the sample custom goal with your own fictional project goal. Task 2: save a second useful note and query both notes through MCP; reconnect and show them again.

In [ ]:
# STUDENT TASK 1: replace this fictional goal with your own project goal.
MY_GOAL_ID = "MY-GOAL-01"
MY_GOAL_TITLE = "Explain one SQLite query to a teammate"
MY_TOPIC = "SQL"
MY_NOTE = "A placeholder keeps a query value separate from SQL instructions."
async with restarted_client:
    my_goal = (await restarted_client.call_tool("remember_goal", {
        "goal_id": MY_GOAL_ID, "title": MY_GOAL_TITLE, "topic": MY_TOPIC})).data
    my_note = (await restarted_client.call_tool("add_learning_note", {"goal_id": MY_GOAL_ID, "note": MY_NOTE})).data
    my_topic_goals = (await restarted_client.call_tool("list_goals", {"topic": MY_TOPIC})).data
print("Your result:", my_goal)
print("Your topic:", my_topic_goals)
assert my_goal["ok"] and my_note["ok"]
assert any(goal["goal_id"] == MY_GOAL_ID for goal in my_topic_goals["goals"])


## 9 Add a useful update tool

Task 3: add reopen_goal to restarted_server. Use server-side validation and a parameterised update. Rediscover and invoke your new capability through MCP.

In [ ]:
# STUDENT TASK 3: register reopen_goal(goal_id) on restarted_server below.
# It should update a known goal's status to planned using a SQL placeholder.
# An unknown ID must return a truthful not-found result and leave other records alone.

async with restarted_client:
    final_tools = await restarted_client.list_tools()
    reopen_added = "reopen_goal" in {tool.name for tool in final_tools}
    if reopen_added:
        reopened = (await restarted_client.call_tool("reopen_goal", {"goal_id": MY_GOAL_ID})).data
        print("Your extension result:", reopened)
    else:
        print("Extension not added yet. Implement reopen_goal above and rediscover it through MCP.")


## 10 Record your observed result

Capture your new record, the read after reconnect, and one result from your added tool. The baseline checks below do not claim that the student extension is finished.

In [ ]:
evidence = {"execution": "ACTUAL_MCP", "transport": "in-memory", "model_calls": 0,
    "new_server_instance": restarted_server is not study_server,
    "saved_goal_status_after_restart": saved_goal["status"],
    "saved_note_read_after_restart": any(row["note"] == DEMO_NOTE for row in notes_after_restart["notes"]),
    "unknown_id_rejected_without_change": rejected["ok"] is False and before == after,
    "resource_read": str(memory_resources[0].uri), "prompt_retrieved": memory_prompts[0].name,
    "new_tool_added_by_student": reopen_added}
print(json.dumps(evidence, indent=2))
print("Core checks passed. Explain persistence and complete your student tasks before submitting.")


## Pair checkpoint and submission

Explain which state lives in Python, which state lives in SQLite and which operation crosses the MCP connection. Submit your notebook, your fictional goal and two notes, a successful read after a new connection, and a real `reopen_goal` tool call. Optional after class: export the database from Colab or connect an LLM after understanding the protocol and memory boundaries.
